# Launch EMR and Conduct CGSS–NDVI Regressions

Run this notebook from your **project root**:

This notebook:

1. Uses `CGSS2018/CGSS2018_clean_main.csv`.
2. Combines local province-level annual NDVI outputs from `ndvi_output/`.
3. Uploads the regression inputs and regression script to `s3://final-project-ndvi/regression/`.
4. Launches an EMR Spark cluster.
5. Submits `code/run_regressions_emr.py` as a Spark step: regress Happiness/WTP on NDVI.
6. Downloads the regression outputs.

Place the companion script `run_regressions_emr.py` inside your local `code/` folder before running the EMR-submit cells.


In [1]:
from pathlib import Path
import os
import json
import time
import datetime as dt

import boto3
import pandas as pd
from botocore.exceptions import ClientError

pd.set_option("display.max_columns", 100)


## 1. Project paths


In [6]:
PROJECT_ROOT = Path.cwd()

CODE_DIR = PROJECT_ROOT / "code"
BOUNDARY_DIR = PROJECT_ROOT / "boundary_data"
MANIFEST_DIR = PROJECT_ROOT / "manifests"
TASKS_DIR = PROJECT_ROOT / "tasks_csv"
LOCAL_NDVI_DIR = PROJECT_ROOT / "ndvi_outputs"
CGSS_DIR = PROJECT_ROOT / "CGSS2018"
REGRESSION_LOCAL_DIR = PROJECT_ROOT / "regression_data"
REGRESSION_LOCAL_DIR.mkdir(exist_ok=True)

CGSS_CLEAN = CGSS_DIR / "CGSS2018_clean_main.csv"
REGRESSION_SCRIPT = CODE_DIR / "run_regressions_emr.py"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CGSS_CLEAN:", CGSS_CLEAN)
print("LOCAL_NDVI_DIR:", LOCAL_NDVI_DIR)
print("REGRESSION_SCRIPT:", REGRESSION_SCRIPT)

required = [CODE_DIR, CGSS_CLEAN, LOCAL_NDVI_DIR]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required paths:\\n" + "\\n".join(missing))

if not REGRESSION_SCRIPT.exists():
    print("\\nWARNING: code/run_regressions_emr.py not found yet.")
    print("Download the companion script and place it here before submitting the EMR step:")
    print(REGRESSION_SCRIPT)


PROJECT_ROOT: c:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness
CGSS_CLEAN: c:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\CGSS2018\CGSS2018_clean_main.csv
LOCAL_NDVI_DIR: c:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\ndvi_outputs
REGRESSION_SCRIPT: c:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\code\run_regressions_emr.py


## 2. AWS and EMR configuration


In [7]:
S3_BUCKET = "final-project-ndvi"

# For AWS Academy coursework, us-east-1 is usually convenient for EMR.
# Change this if your EMR setup is in another region.
EMR_REGION = "us-east-1"
S3_REGION = EMR_REGION

RELEASE_LABEL = "emr-6.15.0"
MASTER_INSTANCE_TYPE = "m5.xlarge"
CORE_INSTANCE_TYPE = "m5.xlarge"
CORE_INSTANCE_COUNT = 2

EC2_KEY_NAME = "ndvi-key"
AUTO_TERMINATION_IDLE_SECONDS = 3600

RUN_ID = dt.datetime.now().strftime("run_%Y%m%d_%H%M%S")

S3_PREFIX = "regression"
S3_CGSS = f"s3://{S3_BUCKET}/{S3_PREFIX}/input/cgss/CGSS2018_clean_main.csv"
S3_NDVI = f"s3://{S3_BUCKET}/{S3_PREFIX}/input/ndvi/ndvi_province_annual_2018.csv"
S3_SCRIPT = f"s3://{S3_BUCKET}/{S3_PREFIX}/code/run_regressions_emr.py"
S3_BOOTSTRAP = f"s3://{S3_BUCKET}/{S3_PREFIX}/code/install_regression_python_packages.sh"
S3_OUTPUT = f"s3://{S3_BUCKET}/{S3_PREFIX}/output/{RUN_ID}"
S3_LOG_URI = f"s3://{S3_BUCKET}/{S3_PREFIX}/emr-logs/"

print("RUN_ID:", RUN_ID)
print("S3_CGSS:", S3_CGSS)
print("S3_NDVI:", S3_NDVI)
print("S3_SCRIPT:", S3_SCRIPT)
print("S3_OUTPUT:", S3_OUTPUT)


RUN_ID: run_20260526_120609
S3_CGSS: s3://final-project-ndvi/regression/input/cgss/CGSS2018_clean_main.csv
S3_NDVI: s3://final-project-ndvi/regression/input/ndvi/ndvi_province_annual_2018.csv
S3_SCRIPT: s3://final-project-ndvi/regression/code/run_regressions_emr.py
S3_OUTPUT: s3://final-project-ndvi/regression/output/run_20260526_120609


In [8]:
def client(service, region=EMR_REGION):
    return boto3.client(service, region_name=region)

def parse_s3_uri(uri):
    assert uri.startswith("s3://"), uri
    no_scheme = uri[5:]
    bucket, _, key = no_scheme.partition("/")
    return bucket, key

def ensure_bucket(bucket_name, region):
    s3 = client("s3", region)
    try:
        s3.head_bucket(Bucket=bucket_name)
        print(f"Bucket exists: s3://{bucket_name}")
        return
    except ClientError:
        pass

    if region == "us-east-1":
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region}
        )
    print(f"Created bucket: s3://{bucket_name}")

def upload_file(local_path, s3_uri):
    local_path = Path(local_path)
    if not local_path.exists():
        raise FileNotFoundError(local_path)
    bucket, key = parse_s3_uri(s3_uri)
    boto3.client("s3").upload_file(str(local_path), bucket, key)
    print(f"Uploaded {local_path} -> {s3_uri}")

def s3_ls(prefix_uri, max_keys=20):
    bucket, prefix = parse_s3_uri(prefix_uri)
    s3 = boto3.client("s3")
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix, MaxKeys=max_keys)
    for obj in resp.get("Contents", []):
        print(obj["Key"], obj["Size"])
    if resp.get("KeyCount", 0) == 0:
        print("(no objects)")


In [9]:
sts = client("sts", EMR_REGION)
identity = sts.get_caller_identity()
print(json.dumps(identity, indent=2))


{
  "UserId": "AROA47CRYUBNG6DOHZ7UZ:user5003845=tingruizhang@uchicago.edu",
  "Account": "891377197146",
  "Arn": "arn:aws:sts::891377197146:assumed-role/voclabs/user5003845=tingruizhang@uchicago.edu",
  "ResponseMetadata": {
    "RequestId": "bbef2294-09c4-48b6-b6cf-8eb93134e1a5",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "bbef2294-09c4-48b6-b6cf-8eb93134e1a5",
      "x-amz-sts-extended-request-id": "MTp1cy1lYXN0LTE6UzoxNzc5ODE1MTg2MzIwOlI6eDFnS0kwS2Q=",
      "content-type": "text/xml",
      "content-length": "488",
      "date": "Tue, 26 May 2026 17:06:26 GMT"
    },
    "RetryAttempts": 0
  }
}


## 3. Combine local NDVI annual outputs


In [21]:
def normalize_province_from_path(path: Path):
    parent = path.parent.name.strip().lower()
    stem = path.stem.strip().lower()

    if parent and parent != "ndvi_output":
        return parent

    for suffix in ["_annual_ndvi_2018", "_annual_ndvi", "_ndvi_2018"]:
        if stem.endswith(suffix):
            return stem.replace(suffix, "")

    return stem.split("_annual")[0]

def find_first_col(columns, candidates):
    cols = list(columns)
    lower = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    for c in cols:
        lc = c.lower()
        for cand in candidates:
            if cand.lower() in lc:
                return c
    return None

annual_files = sorted([
    p for p in LOCAL_NDVI_DIR.rglob("*.csv")
    if "annual" in p.name.lower() and "2018" in p.name.lower()
])

print(f"Found {len(annual_files)} annual NDVI files")
for p in annual_files[:10]:
    print(" ", p.relative_to(PROJECT_ROOT))
if len(annual_files) > 10:
    print(" ...")

if not annual_files:
    raise FileNotFoundError("No annual 2018 NDVI CSV files found under ndvi_output/")

rows = []
for path in annual_files:
    df = pd.read_csv(path)
    if df.empty:
        print("Skipping empty file:", path)
        continue

    province = normalize_province_from_path(path)

    year_col = find_first_col(df.columns, ["year"])
    ndvi_col = find_first_col(
        df.columns,
        ["mean_ndvi", "annual_mean_ndvi", "avg_ndvi", "ndvi_mean", "ndvi", "mean"]
    )

    if ndvi_col is None:
        print("Could not find NDVI column in:", path)
        print("Columns:", df.columns.tolist())
        continue

    if year_col is not None:
        d2018 = df[pd.to_numeric(df[year_col], errors="coerce") == 2018]
        df_use = d2018 if not d2018.empty else df
    else:
        df_use = df

    mean_ndvi = pd.to_numeric(df_use[ndvi_col], errors="coerce").mean()

    rows.append({
        "province": province,
        "year": 2018,
        "mean_ndvi": mean_ndvi,
        "source_file": str(path.relative_to(PROJECT_ROOT)),
        "source_ndvi_col": ndvi_col,
        "n_source_rows": len(df_use),
    })

ndvi_combined = pd.DataFrame(rows).sort_values("province").reset_index(drop=True)
NDVI_COMBINED_LOCAL = REGRESSION_LOCAL_DIR / "ndvi_province_annual_2018.csv"
ndvi_combined.to_csv(NDVI_COMBINED_LOCAL, index=False)

print("Saved:", NDVI_COMBINED_LOCAL)
display(ndvi_combined)


Found 28 annual NDVI files
  ndvi_outputs\anhui\Anhui_annual_ndvi_2018.csv
  ndvi_outputs\beijing\Beijing_annual_ndvi_2018.csv
  ndvi_outputs\chongqing\Chongqing_annual_ndvi_2018.csv
  ndvi_outputs\fujian\Fujian_annual_ndvi_2018.csv
  ndvi_outputs\gansu\Gansu_annual_ndvi_2018.csv
  ndvi_outputs\guangdong\Guangdong_annual_ndvi_2018.csv
  ndvi_outputs\guangxi\Guangxi_annual_ndvi_2018.csv
  ndvi_outputs\guizhou\Guizhou_annual_ndvi_2018.csv
  ndvi_outputs\hebei\Hebei_annual_ndvi_2018.csv
  ndvi_outputs\heilongjiang\Heilongjiang_annual_ndvi_2018.csv
 ...
Saved: c:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\regression_data\ndvi_province_annual_2018.csv


,province,year,mean_ndvi,source_file,source_ndvi_col,n_source_rows
0,anhui,2018,154.0,ndvi_outputs\anhui\Anhui_annual_ndvi_2018.csv,n_ndvi_files_used,1
1,beijing,2018,73.0,ndvi_outputs\beijing\Beijing_annual_ndvi_2018.csv,n_ndvi_files_used,1
2,chongqing,2018,143.0,ndvi_outputs\chongqing\Chongqing_annual_ndvi_2...,n_ndvi_files_used,1
3,fujian,2018,167.0,ndvi_outputs\fujian\Fujian_annual_ndvi_2018.csv,n_ndvi_files_used,1
4,gansu,2018,2523.0,ndvi_outputs\gansu\Gansu_annual_ndvi_2018.csv,n_ndvi_files_used,1
5,guangdong,2018,330.0,ndvi_outputs\guangdong\Guangdong_annual_ndvi_2...,n_ndvi_files_used,1
6,guangxi,2018,301.0,ndvi_outputs\guangxi\Guangxi_annual_ndvi_2018.csv,n_ndvi_files_used,1
7,guizhou,2018,202.0,ndvi_outputs\guizhou\Guizhou_annual_ndvi_2018.csv,n_ndvi_files_used,1
8,hebei,2018,100.0,ndvi_outputs\hebei\Hebei_annual_ndvi_2018.csv,n_ndvi_files_used,1
9,heilongjiang,2018,1359.0,ndvi_outputs\heilongjiang\Heilongjiang_annual_...,n_ndvi_files_used,1


## 4. Upload regression inputs and code to S3


In [22]:
ensure_bucket(S3_BUCKET, S3_REGION)

upload_file(CGSS_CLEAN, S3_CGSS)
upload_file(NDVI_COMBINED_LOCAL, S3_NDVI)

if REGRESSION_SCRIPT.exists():
    upload_file(REGRESSION_SCRIPT, S3_SCRIPT)
else:
    raise FileNotFoundError(
        f"{REGRESSION_SCRIPT} does not exist. "
        "Download run_regressions_emr.py and place it in code/ first."
    )

bootstrap_local = REGRESSION_LOCAL_DIR / "install_regression_python_packages.sh"
bootstrap_text = '''#!/usr/bin/env bash
set -euxo pipefail
sudo python3 -m pip install --upgrade pip
sudo python3 -m pip install --upgrade pandas numpy statsmodels pyarrow boto3
'''
bootstrap_local.write_text(bootstrap_text, encoding="utf-8")
upload_file(bootstrap_local, S3_BOOTSTRAP)

print("\\nS3 regression folder preview:")
s3_ls(f"s3://{S3_BUCKET}/{S3_PREFIX}/", max_keys=50)


Bucket exists: s3://final-project-ndvi
Uploaded c:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\CGSS2018\CGSS2018_clean_main.csv -> s3://final-project-ndvi/regression/input/cgss/CGSS2018_clean_main.csv
Uploaded c:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\regression_data\ndvi_province_annual_2018.csv -> s3://final-project-ndvi/regression/input/ndvi/ndvi_province_annual_2018.csv
Uploaded c:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\code\run_regressions_emr.py -> s3://final-project-ndvi/regression/code/run_regressions_emr.py
Uploaded c:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\regression_data\install_regression_python_packages.sh -> s3://final-project-ndvi/regression/code/install_regression_python_packages.sh
\nS3 regression folder preview:
regression/code/install_regression_python_packages.sh 162
regression/code/run_regressions_emr.py 

## 5. Choose EMR roles, subnet, and optional key pair


In [12]:
def role_exists(role_name):
    try:
        boto3.client("iam").get_role(RoleName=role_name)
        return True
    except ClientError:
        return False

def instance_profile_exists(profile_name):
    try:
        boto3.client("iam").get_instance_profile(InstanceProfileName=profile_name)
        return True
    except ClientError:
        return False

service_role_candidates = ["EMR_DefaultRole", "EMR_ServiceRole", "LabRole"]
jobflow_role_candidates = ["EMR_EC2_DefaultRole", "LabInstanceProfile", "LabRole"]

SERVICE_ROLE = next((r for r in service_role_candidates if role_exists(r)), "EMR_DefaultRole")
JOB_FLOW_ROLE = next((r for r in jobflow_role_candidates if instance_profile_exists(r)), "EMR_EC2_DefaultRole")

print("SERVICE_ROLE:", SERVICE_ROLE)
print("JOB_FLOW_ROLE / instance profile:", JOB_FLOW_ROLE)

ec2 = client("ec2", EMR_REGION)

vpcs = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])["Vpcs"]
if not vpcs:
    raise RuntimeError("No default VPC found in EMR region.")
vpc_id = vpcs[0]["VpcId"]

subnets = ec2.describe_subnets(
    Filters=[
        {"Name": "vpc-id", "Values": [vpc_id]},
        {"Name": "default-for-az", "Values": ["true"]},
    ]
)["Subnets"]

if not subnets:
    subnets = ec2.describe_subnets(Filters=[{"Name": "vpc-id", "Values": [vpc_id]}])["Subnets"]

subnets = sorted(subnets, key=lambda x: x.get("AvailableIpAddressCount", 0), reverse=True)
SUBNET_ID = subnets[0]["SubnetId"]

print("VPC:", vpc_id)
print("SUBNET_ID:", SUBNET_ID)
print("AZ:", subnets[0]["AvailabilityZone"])

try:
    ec2.describe_key_pairs(KeyNames=[EC2_KEY_NAME])
    EMR_EC2_KEY_NAME = EC2_KEY_NAME
    print("Using key pair:", EMR_EC2_KEY_NAME)
except ClientError:
    EMR_EC2_KEY_NAME = None
    print(f"Key pair {EC2_KEY_NAME!r} not found in {EMR_REGION}. Launching EMR without SSH key.")


SERVICE_ROLE: EMR_DefaultRole
JOB_FLOW_ROLE / instance profile: EMR_EC2_DefaultRole
VPC: vpc-0719ae41763265a8e
SUBNET_ID: subnet-0eecca6c77899f7fd
AZ: us-east-1a
Key pair 'ndvi-key' not found in us-east-1. Launching EMR without SSH key.


## 6. Launch EMR cluster


In [14]:
emr = client("emr", EMR_REGION)

instance_groups = [
    {
        "Name": "Master",
        "Market": "ON_DEMAND",
        "InstanceRole": "MASTER",
        "InstanceType": MASTER_INSTANCE_TYPE,
        "InstanceCount": 1,
    },
    {
        "Name": "Core",
        "Market": "ON_DEMAND",
        "InstanceRole": "CORE",
        "InstanceType": CORE_INSTANCE_TYPE,
        "InstanceCount": CORE_INSTANCE_COUNT,
    },
]

instances_config = {
    "InstanceGroups": instance_groups,
    "KeepJobFlowAliveWhenNoSteps": True,
    "TerminationProtected": False,
    "Ec2SubnetId": SUBNET_ID,
}

# Add SSH key only if it exists in this EMR region
if EMR_EC2_KEY_NAME:
    instances_config["Ec2KeyName"] = EMR_EC2_KEY_NAME

cluster_response = emr.run_job_flow(
    Name=f"cgss-ndvi-regression-{RUN_ID}",
    ReleaseLabel=RELEASE_LABEL,
    Applications=[
        {"Name": "Hadoop"},
        {"Name": "Spark"},
    ],
    Instances=instances_config,
    JobFlowRole=JOB_FLOW_ROLE,
    ServiceRole=SERVICE_ROLE,
    LogUri=S3_LOG_URI,
    BootstrapActions=[
        {
            "Name": "Install Python packages for regression",
            "ScriptBootstrapAction": {
                "Path": S3_BOOTSTRAP,
                "Args": [],
            },
        }
    ],
    AutoTerminationPolicy={
        "IdleTimeout": AUTO_TERMINATION_IDLE_SECONDS
    },
    VisibleToAllUsers=True,
)

CLUSTER_ID = cluster_response["JobFlowId"]
print("CLUSTER_ID:", CLUSTER_ID)

CLUSTER_ID: j-25MD44FFUS1OZ


In [15]:
def wait_for_cluster(cluster_id, target_states=("WAITING", "RUNNING"), poll_seconds=30):
    while True:
        desc = emr.describe_cluster(ClusterId=cluster_id)["Cluster"]
        state = desc["Status"]["State"]
        reason = desc["Status"].get("StateChangeReason", {})
        print(dt.datetime.now().strftime("%H:%M:%S"), state, reason.get("Message", ""))
        if state in target_states:
            return desc
        if state in ["TERMINATED", "TERMINATED_WITH_ERRORS"]:
            raise RuntimeError(f"Cluster ended before ready: {state} {reason}")
        time.sleep(poll_seconds)

cluster_desc = wait_for_cluster(CLUSTER_ID)
cluster_desc["Status"]


12:10:50 STARTING Provisioning Amazon EC2 capacity
12:11:20 STARTING Provisioning Amazon EC2 capacity
12:11:51 BOOTSTRAPPING Running bootstrap actions
12:12:22 BOOTSTRAPPING Running bootstrap actions
12:12:52 BOOTSTRAPPING Running bootstrap actions
12:13:23 BOOTSTRAPPING Running bootstrap actions
12:13:54 BOOTSTRAPPING Running bootstrap actions
12:14:24 BOOTSTRAPPING Running bootstrap actions
12:14:55 WAITING Cluster ready to run steps.


{'State': 'WAITING',
 'StateChangeReason': {'Message': 'Cluster ready to run steps.'},
 'Timeline': {'CreationDateTime': datetime.datetime(2026, 5, 26, 12, 10, 45, 869000, tzinfo=tzlocal()),
  'ReadyDateTime': datetime.datetime(2026, 5, 26, 12, 14, 33, 454000, tzinfo=tzlocal())}}

## 7. Submit regression Spark step


In [23]:
step_args = [
    "spark-submit",
    "--deploy-mode", "client",
    S3_SCRIPT,
    "--cgss-s3", S3_CGSS,
    "--ndvi-s3", S3_NDVI,
    "--output-s3", S3_OUTPUT,
    "--year", "2018",
]

step_response = emr.add_job_flow_steps(
    JobFlowId=CLUSTER_ID,
    Steps=[
        {
            "Name": f"Run CGSS-NDVI regressions {RUN_ID}",
            "ActionOnFailure": "CONTINUE",
            "HadoopJarStep": {
                "Jar": "command-runner.jar",
                "Args": step_args,
            },
        }
    ],
)

STEP_ID = step_response["StepIds"][0]
print("STEP_ID:", STEP_ID)
print("Output will be written to:", S3_OUTPUT)


STEP_ID: s-063256627SRLQJCW54TU
Output will be written to: s3://final-project-ndvi/regression/output/run_20260526_120609


In [24]:
def wait_for_step(cluster_id, step_id, poll_seconds=30):
    while True:
        step = emr.describe_step(ClusterId=cluster_id, StepId=step_id)["Step"]
        state = step["Status"]["State"]
        print(dt.datetime.now().strftime("%H:%M:%S"), state)
        if state == "COMPLETED":
            return step
        if state in ["FAILED", "CANCELLED", "INTERRUPTED"]:
            failure = step["Status"].get("FailureDetails", {})
            raise RuntimeError(f"Step failed: {json.dumps(failure, indent=2)}")
        time.sleep(poll_seconds)

step_final = wait_for_step(CLUSTER_ID, STEP_ID)
step_final["Status"]


12:29:18 PENDING
12:29:49 PENDING
12:30:20 RUNNING
12:30:50 RUNNING
12:31:21 RUNNING
12:31:51 RUNNING
12:32:22 RUNNING
12:32:52 COMPLETED


{'State': 'COMPLETED',
 'StateChangeReason': {},
 'Timeline': {'CreationDateTime': datetime.datetime(2026, 5, 26, 12, 29, 16, 129000, tzinfo=tzlocal()),
  'StartDateTime': datetime.datetime(2026, 5, 26, 12, 29, 43, 930000, tzinfo=tzlocal()),
  'EndDateTime': datetime.datetime(2026, 5, 26, 12, 32, 46, 38000, tzinfo=tzlocal())}}

## 8. Inspect and download regression outputs


In [25]:
print("S3 output preview:")
s3_ls(S3_OUTPUT + "/", max_keys=100)


S3 output preview:
regression/output/run_20260526_120609/analysis_data_compact.csv 937076
regression/output/run_20260526_120609/analysis_data_spark_csv/_SUCCESS 0
regression/output/run_20260526_120609/analysis_data_spark_csv/part-00000-ad48d3f8-0158-4b4a-8078-be6c7e4195c6-c000.csv 2153196
regression/output/run_20260526_120609/merge_diagnostics.csv 132
regression/output/run_20260526_120609/regression_notes.txt 1094
regression/output/run_20260526_120609/regression_results_spark_ml.csv 6777
regression/output/run_20260526_120609/regression_results_statsmodels_clustered.csv 9543
regression/output/run_20260526_120609/unmatched_cgss_provinces.csv 26
regression/output/run_20260526_120609/unmatched_ndvi_provinces.csv 13


In [26]:
LOCAL_OUTPUT_RUN = REGRESSION_LOCAL_DIR / "emr_outputs" / RUN_ID
LOCAL_OUTPUT_RUN.mkdir(parents=True, exist_ok=True)

def download_s3_prefix(s3_prefix, local_dir):
    bucket, prefix = parse_s3_uri(s3_prefix.rstrip("/") + "/")
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")
    count = 0
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            rel = key[len(prefix):]
            if not rel or rel.endswith("/"):
                continue
            out_path = Path(local_dir) / rel
            out_path.parent.mkdir(parents=True, exist_ok=True)
            s3.download_file(bucket, key, str(out_path))
            count += 1
    print(f"Downloaded {count} files to {local_dir}")

download_s3_prefix(S3_OUTPUT, LOCAL_OUTPUT_RUN)

for p in sorted(LOCAL_OUTPUT_RUN.rglob("*")):
    if p.is_file():
        print(p.relative_to(LOCAL_OUTPUT_RUN))


Downloaded 9 files to c:\Users\26540\Desktop\学习\Uchicago\final-project-greenness-infrastructure-happiness\regression_data\emr_outputs\run_20260526_120609
analysis_data_compact.csv
analysis_data_spark_csv\_SUCCESS
analysis_data_spark_csv\part-00000-173e877f-fe53-4df6-8f9d-210c71f6c769-c000.csv
analysis_data_spark_csv\part-00000-ad48d3f8-0158-4b4a-8078-be6c7e4195c6-c000.csv
merge_diagnostics.csv
regression_notes.txt
regression_results_spark_ml.csv
regression_results_statsmodels_clustered.csv
unmatched_cgss_provinces.csv
unmatched_ndvi_provinces.csv


In [27]:
stats_path = LOCAL_OUTPUT_RUN / "regression_results_statsmodels_clustered.csv"
spark_path = LOCAL_OUTPUT_RUN / "regression_results_spark_ml.csv"
diag_path = LOCAL_OUTPUT_RUN / "merge_diagnostics.csv"

if stats_path.exists():
    stats_results = pd.read_csv(stats_path)
    display(stats_results)
else:
    print("Statsmodels result file not found:", stats_path)

if diag_path.exists():
    display(pd.read_csv(diag_path))


,outcome,spec,term,estimate,std_error,p_value,n,r2,cov_type,note
0,happiness,spec0_ndvi_only,const,3.906433,0.038895,2.751780e-36,11061,0.000203,cluster_by_province,NaN
1,happiness,spec0_ndvi_only,mean_ndvi,-0.000023,0.000055,6.863753e-01,11061,0.000203,cluster_by_province,NaN
2,happiness,spec1_demographics,const,3.947794,0.088451,7.742307e-27,11061,0.022741,cluster_by_province,NaN
3,happiness,spec1_demographics,mean_ndvi,0.000019,0.000043,6.710365e-01,11061,0.022741,cluster_by_province,NaN
4,happiness,spec1_demographics,age,-0.017795,0.002791,7.906392e-07,11061,0.022741,cluster_by_province,NaN
...,...,...,...,...,...,...,...,...,...,...
59,ln_wtp_air3,spec3_full_controls,health,0.003180,0.011909,7.914701e-01,9854,0.013105,cluster_by_province,NaN
60,ln_wtp_air3,spec3_full_controls,rural_hukou,-0.084308,0.030894,1.104026e-02,9854,0.013105,cluster_by_province,NaN
61,ln_wtp_air3,spec3_full_controls,married,0.035107,0.037799,3.612399e-01,9854,0.013105,cluster_by_province,NaN
62,ln_wtp_air3,spec3_full_controls,ccp_member,-0.017106,0.041505,6.834783e-01,9854,0.013105,cluster_by_province,NaN


,n_cgss_rows,n_ndvi_provinces,n_merged_rows,n_missing_ndvi_after_merge,n_provinces_cgss,n_provinces_with_ndvi
0,11061,28,11061,0,28,28


## 9. Optional: terminate EMR cluster manually


In [29]:

emr.terminate_job_flows(JobFlowIds=[CLUSTER_ID])
print("Terminated:", CLUSTER_ID)


Terminated: j-25MD44FFUS1OZ
